In [138]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import TargetEncoder,OneHotEncoder,OrdinalEncoder,RobustScaler
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.tree import DecisionTreeRegressor,DecisionTreeClassifier
from sklearn.ensemble import RandomForestRegressor
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import  Pipeline
from sklearn.ensemble import AdaBoostClassifier
from xgboost import XGBClassifier,XGBRegressor
from xgboost import XGBClassifier

In [139]:
df=pd.read_csv('uae_ecom_fraud_100k.csv')

In [140]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 36 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   transaction_id          100000 non-null  int64  
 1   user_id                 100000 non-null  int64  
 2   timestamp_utc           100000 non-null  object 
 3   amount_aed              100000 non-null  float64
 4   currency                100000 non-null  object 
 5   payment_method          100000 non-null  object 
 6   device_type             100000 non-null  object 
 7   browser                 100000 non-null  object 
 8   merchant_category       100000 non-null  object 
 9   items_count             100000 non-null  int64  
 10  avg_item_price          100000 non-null  float64
 11  shipping_city           100000 non-null  object 
 12  billing_city            100000 non-null  object 
 13  shipping_billing_match  100000 non-null  int64  
 14  ip_address           

In [141]:
df.head()

,transaction_id,user_id,timestamp_utc,amount_aed,currency,payment_method,device_type,browser,merchant_category,items_count,...,local_hour,odd_hour,is_fraud,fraud_flag_ip,fraud_flag_mismatch,fraud_flag_velocity,fraud_flag_new_account,fraud_flag_prev_cb,fraud_flag_odd_hour,data_source
0,500086758,1000466,2024-01-01T00:02:03Z,71.47,AED,card,desktop,Chrome,Pharmacy,3,...,4,1,0,0,0,0,0,0,1,synthetic:ecom-fraud-uae-v1
1,500060464,1002828,2024-01-01T00:03:11Z,70.25,AED,card,mobile,Chrome,Groceries,1,...,4,1,0,0,0,0,1,0,1,synthetic:ecom-fraud-uae-v1
2,500039909,1010349,2024-01-01T00:04:10Z,116.25,AED,bank_transfer,desktop,Chrome,Beauty,2,...,4,1,0,0,0,0,0,0,1,synthetic:ecom-fraud-uae-v1
3,500074412,1015360,2024-01-01T00:05:14Z,99.41,AED,apple_pay,mobile,Safari,Toys,2,...,4,1,0,0,0,0,0,0,1,synthetic:ecom-fraud-uae-v1
4,500038110,1022253,2024-01-01T00:11:03Z,529.40,AED,card,desktop,Safari,Travel,2,...,4,1,0,0,0,0,1,1,1,synthetic:ecom-fraud-uae-v1


In [142]:
df.columns

Index(['transaction_id', 'user_id', 'timestamp_utc', 'amount_aed', 'currency',
       'payment_method', 'device_type', 'browser', 'merchant_category',
       'items_count', 'avg_item_price', 'shipping_city', 'billing_city',
       'shipping_billing_match', 'ip_address', 'ip_risk_score', 'card_present',
       'bin_country', 'card_age_days', 'card_country_match', 'email_domain',
       'user_prev_chargebacks', 'user_is_high_risk', 'user_account_age_days',
       'transactions_last_24h', 'transactions_last_1h', 'local_hour',
       'odd_hour', 'is_fraud', 'fraud_flag_ip', 'fraud_flag_mismatch',
       'fraud_flag_velocity', 'fraud_flag_new_account', 'fraud_flag_prev_cb',
       'fraud_flag_odd_hour', 'data_source'],
      dtype='object')

In [143]:
for i in ['transaction_id', 'user_id', 'timestamp_utc', 'amount_aed', 'currency',
       'payment_method', 'device_type', 'browser', 'merchant_category',
       'items_count', 'avg_item_price', 'shipping_city', 'billing_city',
       'shipping_billing_match', 'ip_address', 'ip_risk_score', 'card_present',
       'bin_country', 'card_age_days', 'card_country_match', 'email_domain',
       'user_prev_chargebacks', 'user_is_high_risk', 'user_account_age_days',
       'transactions_last_24h', 'transactions_last_1h', 'local_hour',
       'odd_hour', 'is_fraud', 'fraud_flag_ip', 'fraud_flag_mismatch',
       'fraud_flag_velocity', 'fraud_flag_new_account', 'fraud_flag_prev_cb',
       'fraud_flag_odd_hour', 'data_source']:
       print(i,"--->",df[i].nunique())

transaction_id ---> 100000
user_id ---> 28937
timestamp_utc ---> 99839
amount_aed ---> 33713
currency ---> 1
payment_method ---> 5
device_type ---> 3
browser ---> 6
merchant_category ---> 10
items_count ---> 9
avg_item_price ---> 25205
shipping_city ---> 13
billing_city ---> 13
shipping_billing_match ---> 2
ip_address ---> 99993
ip_risk_score ---> 998
card_present ---> 2
bin_country ---> 7
card_age_days ---> 1930
card_country_match ---> 2
email_domain ---> 10
user_prev_chargebacks ---> 3
user_is_high_risk ---> 2
user_account_age_days ---> 2191
transactions_last_24h ---> 3
transactions_last_1h ---> 2
local_hour ---> 24
odd_hour ---> 2
is_fraud ---> 2
fraud_flag_ip ---> 2
fraud_flag_mismatch ---> 2
fraud_flag_velocity ---> 1
fraud_flag_new_account ---> 2
fraud_flag_prev_cb ---> 2
fraud_flag_odd_hour ---> 2
data_source ---> 1


In [144]:
df=df.drop(columns=['data_source','fraud_flag_velocity','currency','transaction_id'],axis=1)

In [145]:
df['timestamp_utc']=pd.to_datetime(df['timestamp_utc'])

In [146]:
df['timestamp_hour']=df['timestamp_utc'].dt.hour
df['timestamp_minute']=df['timestamp_utc'].dt.minute
df['timestamp_day']=df['timestamp_utc'].dt.day
df['timestamp_month']=df['timestamp_utc'].dt.month
df.drop('timestamp_utc',axis=1,inplace=True)

In [ ]:
x=df.drop('is_fraud',axis=1)
y=df.is_fraud

In [152]:
num_col=x.select_dtypes(include='number').columns
obj_col=x.select_dtypes(exclude='number').columns

without using encoder

In [162]:
df[obj_col]=df[obj_col].astype("category")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 35 columns):
 #   Column                  Non-Null Count   Dtype   
---  ------                  --------------   -----   
 0   user_id                 100000 non-null  int64   
 1   amount_aed              100000 non-null  float64 
 2   payment_method          100000 non-null  category
 3   device_type             100000 non-null  category
 4   browser                 100000 non-null  category
 5   merchant_category       100000 non-null  category
 6   items_count             100000 non-null  int64   
 7   avg_item_price          100000 non-null  float64 
 8   shipping_city           100000 non-null  category
 9   billing_city            100000 non-null  category
 10  shipping_billing_match  100000 non-null  int64   
 11  ip_address              100000 non-null  category
 12  ip_risk_score           100000 non-null  float64 
 13  card_present            100000 non-null  int64   
 14  bin_c

In [163]:
x=df.drop('is_fraud',axis=1)
y=df.is_fraud


In [165]:
xtrain,xtest1,ytrain,ytest=train_test_split(x,y,train_size=0.8,random_state=42)

In [167]:
model=XGBClassifier(enable_categorical=True)

In [168]:
model.fit(xtrain,ytrain)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,True
,eval_metric,None


In [170]:
model.score(xtrain,ytrain)

0.932075

In [171]:
model.score(xtest,ytest)

0.91435

with using encoder

In [ ]:
# num_col=xtrain.select_dtypes(include='number').columns
# obj_col=xtrain.select_dtypes(exclude='number').columns

In [ ]:
# preprocessing=ColumnTransformer(
#     transformers=[
#         ('scaler',RobustScaler(),num_col),
#         ('target_encoder',OrdinalEncoder(handle_unknown='use_encoded_value',unknown_value=-1),obj_col)
#     ]
# )

In [ ]:
Logistic_pipeline=Pipeline(
    steps=[
        ('preprocessing',preprocessing),
        ('model',LogisticRegression())
    ]
)

In [ ]:
Logistic_pipeline.fit(xtrain,ytrain)

c:\Users\Admin\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,steps,"[('preprocessing', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('scaler', ...), ('target_encoder', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [ ]:
Logistic_pipeline.score(xtrain,ytrain)

0.918075

In [ ]:
Logistic_pipeline.score(xtest,ytest)

0.91715

In [ ]:
decisiontree_pipeline=Pipeline(
    steps=[
        ('preprocessing',preprocessing),
        ('model',DecisionTreeClassifier())
    ]
)

In [ ]:
decisiontree_pipeline.fit(xtrain,ytrain)

,steps,"[('preprocessing', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('scaler', ...), ('target_encoder', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [ ]:
decisiontree_pipeline.score(xtrain,ytrain)

1.0

In [ ]:
decisiontree_pipeline.score(xtest,ytest)

0.6346

In [ ]:
XG_boost_pipeline=Pipeline(
    steps=[
        ('preprocessing',preprocessing),
        ('model',XGBClassifier())
    ]
)

In [ ]:
XG_boost_pipeline.fit(xtrain,ytrain)

,steps,"[('preprocessing', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('scaler', ...), ('target_encoder', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [ ]:
XG_boost_pipeline.score(xtrain,ytrain)

0.9291

In [ ]:
XG_boost_pipeline.score(xtest,ytest)

0.914